# Lab 30: Capstone, Your Own AIoT Application

**Duration:** 6 hours (two sessions). **Level:** all earlier labs completed. **Work mode:** pairs (or a team of three).

| Student names | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
This is the last lab. There are no step-by-step instructions. **You** choose a problem, design a solution and build it: an **AIoT application** where the **Jetson** is the brain (it sees, hears or thinks) and the **ESP32-S3** is the body (it switches things and reads sensors). You use what you learned in Labs 0 to 29. The notebook is your **project workbook**: it guides you through planning, a working skeleton, testing and the presentation, and it keeps all your records in one place.

**By the end of this lab you can:**
1. Turn an idea into a clear problem, requirements and a test plan.
2. Design a system with sensing, AI, decision, action and logging, and name its failure modes.
3. Build and test it, and measure it (speed, delay, accuracy, false alarms).
4. Make it safe (fail-safe, privacy) and explain your choices.
5. Present and defend your project.

**How this notebook works.** Work in order. The code cells are **a skeleton that already runs**: a tiny demo that switches the board alert when the camera is covered. Replace each part with your own AI and your own devices. Type your answers in the cells that say *Your answers*. Press **Ctrl + S** often.

> **Board.** Use the **virtual ESP32** (Lab 27) and, when your real board is ready, switch to it. A project that works with the virtual board and **also** with the real one earns the most credit.
> **Memory.** The 8 GB are shared. Use one or two models, not six. Prefer the TensorRT engine for the heavy model.

## 2. The Rules of the Capstone

Your project **must have** all of these. The check task lists them again.

| No. | Requirement |
|---|---|
| 1 | A **real problem** that someone has, written in one sentence |
| 2 | At least **one AI part on the Jetson** from Labs 6 to 26 (detector, tracker, pose, segmentation, OCR/QR, your own trained model, speech, keyword, language model, vision-language model) or a sensor AI of Lab 29 |
| 3 | At least **one IoT part on the ESP32** (LED, buzzer, servo, relay and so on, or a sensor), talking over MQTT (Lab 27) |
| 4 | A **decision rule** that does not flicker (hold time, hysteresis or confirm) and sends commands only when the situation changes |
| 5 | A **log** in CSV or Excel with the time of every event |
| 6 | A **safe state**: what the system does when something fails (the fail-safe of Lab 27) |
| 7 | A **privacy and safety** note: what data it sees, where they go, who could be harmed |
| 8 | A **test table** with at least 8 tests, with numbers (for example accuracy, false alarms, delay) |
| 9 | A **3-minute demonstration** and a short report |

**Marks.** The project is judged on: the problem (10), design (15), AI part (20), IoT part (15), reliability and safety (15), testing and numbers (15), presentation and report (10).

## Project ideas
Pick one or invent your own. Each idea uses the lab that is named.

| Idea | AI on the Jetson (Lab) | IoT on the ESP32 |
|---|---|---|
| **Smart door** for a store room | QR and doorway safety (13, 12, 23) | Servo door, LED |
| **Room guardian**: too many people, too dark or too hot | People count (10), sensor AI (24) | Buzzer, LED, light and temperature sensors |
| **Tool or lab-kit checker** | Your own model and the missing-item alarm (14, 15) | LED per tool, buzzer |
| **Pet or plant helper** | Your own model of the pet or a plant (14) and the camera question (20) | Servo feeder, soil sensor, pump relay |
| **Voice-controlled room** | Keyword and speech (16, 17, 18) | Lamp relay, fan, door servo |
| **Exercise coach** | Pose and rep counter (11) | LED ring or buzzer for feedback |
| **Safety monitor** | Zones and tracking (10, 12) | Alarm, a stop signal for a small machine |
| **Your own idea** | Any | Any |

---
# Part A: The Idea and the Plan (60 min)

### Step 1: The Project Brief
Write the brief with your partner. Be short and precise. A good problem is something you can **measure**: "the store room door is left open" is better than "make the school smart".

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Project name:
- The problem in one sentence (who has the problem?):
- Why is AI useful here? What would a simple sensor or timer not do?:
- What does the Jetson sense (camera, microphone) and what does it decide?:
- What does the ESP32 do (devices and sensors)?:
- How will you know that it works? Write 2 numbers you will measure (for example 'finds the person 9 times out of 10', 'the alarm comes within 2 seconds'):
- What can go wrong and what is the safe state?:
- Privacy and safety: what data are used, who could be harmed, and how do you reduce the risk?:


### Step 2: The Architecture
A block diagram makes the design clear for you and for the people who read your report. `draw_architecture` draws one with OpenCV from lists you write: the boxes **on the Jetson** (blue), the **link** (MQTT topics, orange) and the boxes **on the ESP32** (green). Change the lists to your own design and run the cell until the picture is right.

In [ ]:
import csv
import os
import time
from datetime import datetime

import cv2
import numpy as np
from IPython.display import Image, display

import labview

JETSON = ["Camera", "AI: detector", "Decision (hold time)", "Log (CSV, Excel)"]        # the boxes on the Jetson, in order
TOPICS = ["esp/cmd/alert", "esp/cmd/servo", "esp/telemetry"]                                   # the messages on the link
ESP32 = ["Wi-Fi + MQTT", "LED and buzzer", "Servo", "Sensor"]                                  # the boxes on the board


def draw_architecture(jetson, topics, esp32, title="My AIoT system"):
    """A simple block diagram: the Jetson column, the MQTT link, the ESP32 column."""
    rows = max(len(jetson), len(esp32), len(topics))
    height = 90 + rows * 70
    picture = np.full((height, 760, 3), 255, np.uint8)
    cv2.putText(picture, title, (14, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (40, 40, 40), 2)
    cv2.putText(picture, "JETSON (AI brain)", (30, 62), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (160, 90, 0), 2)
    cv2.putText(picture, "MQTT", (350, 62), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 120, 220), 2)
    cv2.putText(picture, "ESP32-S3 (IoT body)", (520, 62), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (40, 140, 40), 2)
    for i, text in enumerate(jetson):
        y = 80 + i * 70
        cv2.rectangle(picture, (20, y), (250, y + 48), (250, 235, 215), -1)
        cv2.rectangle(picture, (20, y), (250, y + 48), (160, 90, 0), 2)
        cv2.putText(picture, text[:26], (30, y + 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
        if i < len(jetson) - 1:
            cv2.arrowedLine(picture, (135, y + 48), (135, y + 70), (160, 90, 0), 2, tipLength=0.4)
    for i, text in enumerate(topics):
        y = 92 + i * 70
        cv2.putText(picture, text[:22], (290, y + 24), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 120, 220), 1)
        cv2.arrowedLine(picture, (285, y + 34), (450, y + 34), (0, 120, 220), 2, tipLength=0.05)
    for i, text in enumerate(esp32):
        y = 80 + i * 70
        cv2.rectangle(picture, (510, y), (740, y + 48), (220, 245, 220), -1)
        cv2.rectangle(picture, (510, y), (740, y + 48), (40, 140, 40), 2)
        cv2.putText(picture, text[:26], (520, y + 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
    return picture


diagram = draw_architecture(JETSON, TOPICS, ESP32)
cv2.imwrite("architecture.png", diagram)
display(Image(data=cv2.imencode(".png", diagram)[1].tobytes()))

### Step 3: Failure Modes
Every part can fail. A reliable system plans for it. Fill in the table for **your** project (add rows).

| What fails | What the user sees | What your system does (the safe state) |
|---|---|---|
| The camera is covered or unplugged | | |
| The model misses the object or sees a ghost | | |
| The ESP32 is offline | | |
| The Wi-Fi or the broker is lost | | |
| The Jetson crashes or the notebook stops | | |
| The power is cut | | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Your failure table (the three most serious rows):
- Which failure is the most dangerous for people? What do you do about it?:


---
# Part B: The Skeleton (60 min)

### Step 1: A Working Skeleton
This is the **frame of every project in this course**: *sense, decide, act, log*. It already runs: the "AI" is a very simple brightness measure, and the system switches the board **alert on** when the camera is covered (dark) for 2 seconds, and **off** when it is bright again. You replace the parts one by one:

| Part | Demo | You replace it with |
|---|---|---|
| `sense(frame)` | The average brightness | Your model: people count, a class, a keyword, text, a pose, ... |
| `decide(m, now)` | Dark for `HOLD` seconds, with two limits | Your own rule with hold time and hysteresis |
| `act(command)` | `esp.send("alert", ...)` | Your own commands: `led`, `buzzer`, `servo`, and others |
| `log_event(...)` | One CSV line per change | The columns your project needs |

Keep the **two rules** of Labs 12 and 27: the decision must not flicker, and a command is sent only when the situation **changes**.

In [ ]:
board = labview.VirtualEsp32()         # the pretend board. For the real board: board.stop() and state["virtual"] = False (Part D of Lab 27)
esp = labview.Esp32()
time.sleep(1)
print("Board online:", esp.online())

# ---------------- Settings of your project ----------------
DARK_AT = 60          # the demo: alert when the brightness is below this ...
BRIGHT_AT = 80        # ... and off again when it is above this (two limits: hysteresis)
HOLD = 2.0            # seconds a new situation must last
LOG_FILE = "capstone_log.csv"
# ----------------------------------------------------------
state = {"on": False, "candidate": None, "since": 0.0, "sent": 0, "note": ""}


def sense(frame):
    """SENSE: turn a picture into measurements. Replace this with your AI. Return a dictionary of numbers."""
    return {"brightness": float(frame.mean())}


def decide(measure, now):
    """DECIDE: return "on" or "off" when the alert must change, otherwise None. Hysteresis and hold time keep it calm."""
    value = measure["brightness"]
    wanted = state["on"]
    if value < DARK_AT:
        wanted = True
    elif value > BRIGHT_AT:
        wanted = False
    if wanted == state["on"]:
        state["candidate"] = None
        return None
    if state["candidate"] != wanted:
        state["candidate"], state["since"] = wanted, now
        return None
    if now - state["since"] >= HOLD:
        state.update(on=wanted, candidate=None)
        return "on" if wanted else "off"
    return None


def act(command):
    """ACT: send the command to the board. Only called when the situation changes. Returns True if it was sent."""
    if not esp.online():
        return False                                  # do not send into the void: the log says so
    esp.send("alert", command)
    return True


def log_event(command, measure, sent):
    """LOG: one line for every change."""
    first = not os.path.exists(LOG_FILE)
    with open(LOG_FILE, "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "command", "measurement", "sent_to_board", "latency_ms"])
        latency = round(1000 * esp.latencies[-1], 1) if esp.latencies and sent else ""
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), command, round(list(measure.values())[0], 1), sent, latency])


def process(frame, state_live):
    """The live view calls this for every picture: sense, decide, act, log, draw."""
    shown = frame.copy()
    measure = sense(frame)
    command = decide(measure, time.time())
    if command:
        sent = act(command)
        state["sent"] += sent
        log_event(command, measure, sent)
        state["note"] = f"{command} {'sent' if sent else 'NOT sent (board offline)'} at {datetime.now().strftime('%H:%M:%S')}"
    colour = (0, 0, 255) if state["on"] else (0, 200, 0)
    cv2.putText(shown, f"ALERT {'ON' if state['on'] else 'off'}   " + "  ".join(f"{k}: {v:.0f}" for k, v in measure.items()),
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, colour, 2)
    link = "board OFFLINE"
    if esp.online():
        link = f"board ONLINE   sent {state['sent']}   " + (board.describe() if state.get("virtual", True) else "")
    cv2.putText(shown, link, (10, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    if state["note"]:
        cv2.putText(shown, state["note"], (10, shown.shape[0] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cv2.putText(shown, f"{state_live['fps']:.0f} FPS", (shown.shape[1] - 110, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    return shown


print("The skeleton is ready.")

### Step 2: Check the Logic Without the Camera
Before any live test, check `decide` with a made-up list of brightness values, one per second. This kind of **test without hardware** is how you find mistakes fast. The demo should send `on` once and `off` once.

In [ ]:
state.update(on=False, candidate=None)
values = [120, 110, 50, 40, 45, 40, 90, 100, 100, 100]
sequence = [(t, decide({"brightness": v}, float(t))) for t, v in enumerate(values)]
print("Brightness:", values)
print("Commands:  ", [(t, c) for t, c in sequence if c], "(expected: on after about 2 s of dark, off after 2 s of bright)")
state.update(on=False, candidate=None, sent=0, note="")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did the demo send exactly one `on` and one `off`?:
- Write the test sequence you will use for **your** decision rule (the values and the commands you expect).:


### Step 3: Run the Demo
Cover the camera with your hand for 3 seconds: the alert goes on (in the picture and on the board). Take your hand away: it goes off after 2 seconds. Press **Stop** when you are done. Then you know the whole chain works, and you can replace the parts.

In [ ]:
state.update(on=False, candidate=None, sent=0, note="")
esp.send("alert", "off")
live = labview.live(process)

After you pressed Stop:

In [ ]:
esp.send("alert", "off")
if os.path.exists(LOG_FILE):
    print(open(LOG_FILE).read())

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did the demo work? FPS and the latency of the command (from the log):
- What is the first part you will replace and with what?:


---
# Part C: Build Your Project (150 min, two sessions)

Replace the parts of the skeleton with your own. Work in **small steps** and **test after each step**: first `sense` (does it measure what you want?), then `decide` (does it not flicker?), then `act` (does the board do the right thing?), then the log. Use the cells below to keep your **build diary**: write what you did and what you measured at each step. A diary helps you find mistakes, and it is part of your mark.

**Tools you can reuse**

| Need | Where |
|---|---|
| Detector and counting, tracking | Labs 9 to 14 |
| Pose, segmentation and zones | Labs 12 and 13 |
| QR codes and text | Labs 7 and 8 |
| Your own model, and the engine | Labs 15 and 32 |
| Speech, keyword, voice | Labs 17 to 19 |
| Language and vision-language models | Labs 20 to 26 |
| MQTT, the board, the fail-safe | Lab 27 |
| Access rules, lockout, state machine | Lab 28 |
| Sensor data, SQL, anomalies, dashboard | Lab 29 |

You can copy the code of the earlier notebooks: open them from the file browser (`lab14`, `lab32`, and so on).

> **Tip.** Change one thing at a time. To keep a copy of a cell before a big change, copy the cell (select it, press **C**, then **V**) and change the copy. Press **Ctrl + S** often.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Session 1, step 1: `sense` (what did you build and what did you measure?):
- Session 1, step 2: `decide` (your rule, the limits, the hold time, and your test sequence result):
- Session 1, step 3: `act` and the board (which commands, which topics, what the board does):
- Session 2, step 4: the log and the dashboard or the display (what do you record?):
- Session 2, step 5: the safe state (what did you build for each failure of your table?):
- The biggest problem you met and how you solved it:


---
# Part D: Test and Measure (60 min)

### Step 1: The Test Table
Test your **whole system** (not only the parts). Write at least **8 tests with numbers**. Include normal cases, hard cases (dim light, noise, two people, wrong object) and **failure cases** (board offline, camera covered, notebook stopped). Measure:

| Metric | How |
|---|---|
| **Accuracy** | Do 20 trials: how many did the system get right? |
| **False alarms** | How many wrong alarms in 5 minutes with nothing happening? |
| **Missed events** | How many real events did it miss in 20 trials? |
| **Delay** | The time from the event to the action of the board (from the log, and the command latency) |
| **Speed** | The FPS of the live view |

Use this table (copy it into the answer cell and fill it in):

| No. | Situation | Expected | Result with numbers |
|---|---|---|---|
| 1 | Normal case | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |
| 5 | A failure case | | |
| 6 | | | |
| 7 | | | |
| 8 | | | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Your test table (all 8 or more lines):
- Accuracy, false alarms, missed events, delay and FPS:
- What is the weakest point of your system and how would you improve it with more time?:


### Step 2: Power and Safety
Run `!sudo nvpmodel -q` (after `labview.sudo_login()`) and `jtop` as in Lab 15. Write the power mode and the temperature while your project runs. Think about the **safety** of the real world: does your system ever make something dangerous (a door, a motor, a heater)? What is its safe state?

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Power mode, memory use and temperature while the project runs:
- The safety risks and your answers to them:


---
# Part E: Present (30 min)

### Step 1: The Report
Fill in the final report. It must be short and clear.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Project name and team:
- The problem and the users:
- The design: what the Jetson does (AI) and what the ESP32 does (IoT), and the MQTT topics:
- The AI part: the model, why you chose it, and the numbers (accuracy, FPS):
- The decision rule and why it does not flicker:
- The safe state and the privacy note:
- The results of the tests (the most important numbers):
- What you learned and what you would change:


### Step 2: The 3-Minute Demonstration
Plan your demonstration. In 3 minutes: **(1)** the problem in one sentence, **(2)** the diagram, **(3)** the live system doing its job, **(4)** one failure and the safe state, **(5)** your best numbers. Practise it once with a timer. Everybody in the team speaks.

> **Congratulations.** You started this course by setting up a small computer. You now built an AI product that sees or hears, decides, controls a device over the network, records what it did, and fails safely. That is what an AIoT engineer does.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Requirement | Initials |
|---|---|---|
| 1 | The problem is clear and written in one sentence | |
| 2 | The architecture diagram is made and matches the system | |
| 3 | An AI part from Labs 6 to 29 runs on the Jetson | |
| 4 | An IoT part works on the ESP32 (virtual or real) over MQTT | |
| 5 | The decision rule does not flicker and commands are sent only on changes | |
| 6 | The log has the time of every event and is shown in Excel | |
| 7 | The safe state works (for example the board is offline, or the notebook is stopped) | |
| 8 | The privacy and safety note is written | |
| 9 | The test table has at least 8 tests with numbers, and the report and the 3-minute demonstration are done | |

## 5. Assessment Questions
Answer in one or two sentences each, in your own words.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- 1. Describe your project in three sentences for a person who does not know AI.:
- 2. Which part is AI and which is IoT? Why is that the right split for your project?:
- 3. How does your system avoid flicker and avoid sending too many messages?:
- 4. What does your system do when the board is offline, and when the Jetson stops?:
- 5. Which test number surprised you the most and why?:
- 6. What is the biggest risk of your system to people or their privacy?:
- 7. If you had the real board only for one more hour, what would you test first?:
- 8. Which lab of this course helped you most in this project and why?:
- 9. Name one thing in your project that a bigger budget (a bigger model, a better camera, more boards) would improve.:


## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| The skeleton does not run | An earlier cell was not run, or the lab was not started | Run the cells from the top. Ask your teacher to start the lab, then restart the kernel. |
| `board OFFLINE` | The board is not running or not on your broker | Run the first cell of Part B again. Check Lab 27 Part B and Part D. |
| The demo never switches | The brightness limits do not match your camera | Print `sense(frame)` and set `DARK_AT` and `BRIGHT_AT` around the real numbers |
| The system flickers | No hold time or no hysteresis | Raise `HOLD`. Use two limits (`DARK_AT` and `BRIGHT_AT` in the demo). |
| Too many commands | The program sends in every picture | Send only when `decide` returns a command (see the skeleton) |
| `CUDA out of memory` | Too many models or other notebooks | Use the engine, close other notebooks, Kernel, Shut Down All Kernels |
| It is too slow | A big model, a big picture, power mode | Use the FP16 engine (Lab 15), a smaller picture, MAXN SUPER (Lab 15) |
| The real board does not work but the virtual one does | Wi-Fi, `config.py` or the wiring | Lab 27 Part D and the troubleshooting table there |
| You break the notebook | Too many changes at once | Close it and run `import labview; labview.reset(26)` in a cell of a new notebook (File, New, Notebook). Your old copy is kept as `.old`. |

## 7. Further Reading
- Your own notebooks of Labs 6 to 29 are the best manual.
- NVIDIA Jetson AI Lab: https://www.jetson-ai-lab.com/
- Ultralytics documentation: https://docs.ultralytics.com/
- MQTT: https://mqtt.org/
- MicroPython for the ESP32: https://docs.micropython.org/en/latest/esp32/quickref.html